In [1]:
import rasterio
from rasterio import Affine
from osgeo import gdal
import geopandas as gp
import shapely
import shapelysmooth as ss
from multiprocessing import Queue
import multiprocessing as mp
import numpy as np
from flight_path import FlightPath
from copy import deepcopy


In [2]:
gdal.UseExceptions()
data : gdal.Dataset = gdal.Open(
    "/home/hugh/SKYHOG/FL_PANHANDLE_LAZ_PROCESSED/ST_MARKS_FULL_MOSAIC.tif"
)

gdal_transformations = data.GetGeoTransform()

extent = data.GetExtent()

transform = Affine(
            gdal_transformations[0],
            gdal_transformations[1],
            gdal_transformations[2],
            gdal_transformations[3],
            gdal_transformations[4],
            gdal_transformations[5]
    )

In [3]:
height_data : rasterio.DatasetReader = rasterio.open(
    "/home/hugh/SKYHOG/FL_PANHANDLE_LAZ_PROCESSED/ST_MARKS_FULL_MOSAIC.tif"
)

height_matrix : np.ndarray = height_data.read(1) #this turns our raw raster data into a np arr where the values are heights above surface

VIEWSHED_DIST = 200 #this is the circle around a node we use to impute its cost

In [4]:
# assumed 2d np arr
def processSegment(seg: np.ndarray, res_queue: Queue , transform : Affine , viewshed : int , buffer_size : int = 0 ) -> None:

    pathfinder = FlightPath(
        data=seg,
        crs=3857,
        viewshed_distance=viewshed,
        resolution_reduction_factor=50,
        avoid_value=-32768,
        transform=transform
    )

    df = pathfinder.calculateFlightPath(included_buffer=buffer_size)

    res_queue.put(df , block=False)

    return

In [5]:
#chunk the array into 200m block segments running E->W
#we need to provide a buffer on these arrays as when we compute the circle window we will run into edge effects otherwise.
#reccomended buffer is same as viewshed distance.

BUFFER = VIEWSHED_DIST

remainder = ( height_matrix.shape[0] % VIEWSHED_DIST , height_matrix.shape[1] % VIEWSHED_DIST )

vertical_divisions = height_matrix.shape[0] // (VIEWSHED_DIST*2)
horizontal_divisions = height_matrix.shape[1] // (VIEWSHED_DIST*2)

segments = []

offset = (VIEWSHED_DIST + BUFFER)

#intentional -1 here, want the second to last
for i in range( 0 , vertical_divisions - 1 ) :
    segments.append(
        deepcopy(height_matrix[ max(offset - BUFFER - VIEWSHED_DIST , 0 , ):offset+BUFFER+VIEWSHED_DIST+ (remainder[0] if i == vertical_divisions - 2 else 0) , : ])
    )

    offset+=VIEWSHED_DIST*2


In [ ]:
#a small multiprocess example

queue = Queue(maxsize=len(segments))

procs : list[mp.Process] = []

for i in range(len(segments)) :

    proc = mp.Process( 
        target=processSegment , 
        args=( 
            segments[i] , 
            queue,    
            Affine( 
            (transform.a ) , 
            transform.b , 
            transform.c , 
            (transform.d - ((VIEWSHED_DIST*2) * i) ) , 
            transform.e , 
            transform.d ) 
            ,
            VIEWSHED_DIST,
            BUFFER
        )
    )

    procs.append(proc)

    proc.start()


flight_lines : list[gp.GeoDataFrame] = []

finished = 0 

while finished != len(procs):
    flight_lines.append(queue.get())

    finished += 1

for proc in procs:
    proc.join()

In [7]:
lines = [ shapely.geometry.LineString(x.geometry) for x in flight_lines ]

In [8]:
lines.sort(key=lambda x : x.coords[0][1])

In [9]:
modified_lines = []

prev = None

for line in lines:

    coords = list(line.coords)
    coords.sort(key=lambda x : x[0])

    if len(modified_lines) == 0:
        prev = coords[0]
        modified_lines.append(coords)
        continue

    if len(modified_lines) % 2 == 0:
        coords.append(prev)
        prev = coords[0]
        modified_lines.append(coords)
    else:
        coords.insert(0 , prev)
        prev = coords[len(coords) - 1]
        modified_lines.append(coords)


In [10]:
flight_line : shapely.LineString = shapely.ops.linemerge(modified_lines)
flight_line = ss.catmull_rom_smooth( flight_line)

In [11]:
gp.GeoDataFrame(data=[] , geometry=[flight_line] , crs=4326).to_file('bar.geojson')